# Day 032 — Exercise 5: SecureConfig

**What you'll build:** The `SecureConfig` class — `load_dict` to layer config from any dict, `get` for optional keys, `require` for mandatory keys (raises on absence), `validate` for startup checking, and `masked_dict` for log-safe output.

**Why it matters:** SecureConfig is a reusable config module you can drop into any automation project. One import, one class, full secret hygiene.

## Provided: All Helper Functions

In [ ]:
def parse_dotenv(text: str) -> dict:
    result = {}
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if "=" not in line:
            continue
        key, _, value = line.partition("=")
        key   = key.strip()
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in ('"', "'"):
            value = value[1:-1]
        result[key] = value
    return result


def mask_secret(value, show_chars: int = 4) -> str:
    s = str(value)
    if len(s) <= show_chars:
        return "***"
    return s[:show_chars] + "***" 


def validate_config(config: dict, required_keys: list) -> list:
    return [k for k in required_keys if config.get(k) is None]


def safe_log_config(config: dict, secret_keys: list) -> dict:
    return {
        k: mask_secret(str(v)) if k in secret_keys else v
        for k, v in config.items()
    }

## Your Implementation

In [ ]:
class SecureConfig:
    """
    Secure config module. Loads from layered dicts (defaults, .env, os.environ).
    Provides safe access via get/require and logging-safe output via masked_dict.
    """

    def __init__(self, defaults: dict | None = None):
        # TODO: self._config = dict(defaults or {})
        pass

    def load_dict(self, mapping: dict) -> 'SecureConfig':
        # TODO: self._config.update(mapping); return self
        pass

    def get(self, key: str, default=None):
        # TODO: return self._config.get(key, default)
        pass

    def require(self, key: str) -> str:
        # TODO: val = self._config.get(key)
        # TODO: if val is None: raise KeyError(f"Required config key not found: '{key}'")
        # TODO: return str(val)
        pass

    def validate(self, required_keys: list) -> list:
        # TODO: return validate_config(self._config, required_keys)
        pass

    def masked_dict(self, secret_keys: list) -> dict:
        # TODO: return safe_log_config(self._config, secret_keys)
        pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: class defined with required methods
    try:
        assert 'SecureConfig' in globals()
        for m in ('load_dict', 'get', 'require', 'validate', 'masked_dict'):
            assert hasattr(SecureConfig, m), f'missing method: {m}'
        passed += 1; print('\u2705 Check 1: SecureConfig with all 5 methods')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}')
        return

    # Check 2: load_dict updates config and returns self (fluent)
    try:
        cfg = SecureConfig(defaults={'MODEL': 'llama3.2'})
        ret = cfg.load_dict({'API_KEY': 'sk-demo', 'DB_HOST': 'localhost'})
        assert ret is cfg, f'load_dict should return self, got {type(ret)}'
        assert cfg.get('MODEL')   == 'llama3.2', f"MODEL wrong: {cfg.get('MODEL')!r}"
        assert cfg.get('API_KEY') == 'sk-demo',  f"API_KEY wrong: {cfg.get('API_KEY')!r}"
        assert cfg.get('DB_HOST') == 'localhost', f"DB_HOST wrong: {cfg.get('DB_HOST')!r}"
        passed += 1; print('\u2705 Check 2: load_dict updates config and returns self')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: get returns value or default
    try:
        cfg2 = SecureConfig({'A': '1', 'B': None})
        assert cfg2.get('A')            == '1',       f"A: {cfg2.get('A')!r}"
        assert cfg2.get('B')            is None,      f"B: {cfg2.get('B')!r}"
        assert cfg2.get('MISSING')      is None,      f"MISSING: {cfg2.get('MISSING')!r}"
        assert cfg2.get('MISSING', 42)  == 42,        f"default: {cfg2.get('MISSING', 42)!r}"
        passed += 1; print('\u2705 Check 3: get returns value or default')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: require raises KeyError on None or absent key
    try:
        cfg3 = SecureConfig({'PRESENT': 'hello', 'NULL_KEY': None})
        assert cfg3.require('PRESENT') == 'hello', \
            f"require present key: {cfg3.require('PRESENT')!r}"
        raised_absent = False
        try:
            cfg3.require('MISSING_KEY')
        except KeyError:
            raised_absent = True
        assert raised_absent, 'require should raise KeyError for absent key'
        raised_none = False
        try:
            cfg3.require('NULL_KEY')
        except KeyError:
            raised_none = True
        assert raised_none, 'require should raise KeyError for None value'
        passed += 1; print('\u2705 Check 4: require raises KeyError on absent/None key')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: validate and masked_dict work correctly
    try:
        cfg4 = SecureConfig({'API_KEY': 'sk-abc123', 'HOST': 'localhost'})
        missing = cfg4.validate(['API_KEY', 'HOST', 'WEBHOOK'])
        assert missing == ['WEBHOOK'], \
            f"validate wrong: {missing}"
        safe = cfg4.masked_dict(['API_KEY'])
        assert '***' in safe.get('API_KEY', ''), \
            f"API_KEY should be masked: {safe.get('API_KEY')!r}"
        assert safe.get('HOST') == 'localhost', \
            f"HOST should be unchanged: {safe.get('HOST')!r}"
        passed += 1; print('\u2705 Check 5: validate and masked_dict correct')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
class SecureConfig:
    def __init__(self, defaults: dict | None = None):
        self._config: dict = dict(defaults or {})

    def load_dict(self, mapping: dict) -> "SecureConfig":
        self._config.update(mapping)
        return self

    def get(self, key: str, default=None):
        return self._config.get(key, default)

    def require(self, key: str) -> str:
        val = self._config.get(key)
        if val is None:
            raise KeyError(f"Required config key not found: '{key}'")
        return str(val)

    def validate(self, required_keys: list) -> list:
        return validate_config(self._config, required_keys)

    def masked_dict(self, secret_keys: list) -> dict:
        return safe_log_config(self._config, secret_keys)
```

</details>